# Fine-tuning Modèle Émotions pour SentiFlow

**IMPORTANT:** Avant de lancer, va dans Runtime → Change runtime type → GPU

In [ ]:
# 1. Installation des dépendances
!pip install -q transformers datasets accelerate scikit-learn

In [ ]:
# 2. Vérifier GPU
import torch
print(f"GPU disponible: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# 3. Charger le dataset
from datasets import load_dataset

dataset = load_dataset("dair-ai/emotion")
print(dataset)
print(f"\nLabels: {dataset['train'].features['label'].names}")

In [ ]:
# 4. Charger le modèle et tokenizer
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "xlm-roberta-base"  # Modèle multilingue
num_labels = 6  # sadness, joy, love, anger, fear, surprise

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels
)
print("Modèle chargé!")

In [ ]:
# 5. Tokenizer le dataset
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print("Dataset tokenisé!")

In [ ]:
# 6. Configuration de l'entraînement
from transformers import TrainingArguments, Trainer
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./emotion_model",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    logging_steps=100,
    fp16=True,  # Mixed precision pour GPU
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    compute_metrics=compute_metrics,
)
print("Trainer configuré!")

In [ ]:
# 7. LANCER L'ENTRAINEMENT (~15-30 min avec GPU)
print("Début de l'entraînement...")
trainer.train()
print("Entraînement terminé!")

In [ ]:
# 8. Évaluation finale
results = trainer.evaluate(tokenized_datasets["test"])
print(f"\n=== RESULTATS FINAUX ===")
print(f"Accuracy: {results['eval_accuracy']:.1%}")
print(f"F1-Score: {results['eval_f1']:.1%}")

In [ ]:
# 9. Sauvegarder le modèle
trainer.save_model("./sentiflow_emotion_model")
tokenizer.save_pretrained("./sentiflow_emotion_model")
print("Modèle sauvegardé dans ./sentiflow_emotion_model")

In [ ]:
# 10. Tester le modèle
from transformers import pipeline

classifier = pipeline("text-classification", model="./sentiflow_emotion_model", top_k=None)

LABELS = ["tristesse", "joie", "joie", "colere", "peur", "surprise"]  # Mapping FR

tests = [
    "Je suis tellement heureux aujourd'hui!",
    "I am so angry right now",
    "Je suis triste et déprimé",
    "This is scary and frightening",
    "Wow I didn't expect that!"
]

print("\n=== TESTS ===")
for text in tests:
    result = classifier(text)[0]
    top = max(result, key=lambda x: x['score'])
    label_idx = int(top['label'].split('_')[-1])
    print(f"\n{text}")
    print(f"  → {LABELS[label_idx].upper()} ({top['score']:.1%})")

In [ ]:
# 11. Télécharger le modèle (ZIP)
!zip -r sentiflow_emotion_model.zip sentiflow_emotion_model/

from google.colab import files
files.download('sentiflow_emotion_model.zip')
print("\nTélécharge le fichier ZIP et décompresse-le dans data/models/ de ton projet!")